<a href="https://colab.research.google.com/github/MadhurJain06/Mini/blob/main/ImmunizeModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import IPython

def keep_alive():
    while True:
        time.sleep(60)
        IPython.display.clear_output()
        print(f"Still alive... {time.strftime('%H:%M:%S')}")

import threading
t = threading.Thread(target=keep_alive, daemon=True)
t.start()

In [ ]:
!pip install diffusers transformers accelerate mediapipe torch torchvision \
             fastapi uvicorn pyngrok pillow numpy opencv-python-headless \
             ftfy regex tqdm -q

In [ ]:
from pyngrok import ngrok
import json

NGROK_TOKEN = "3C8oW6HSq4ct8E8LHYrEJ8ihsVJ_5qGuWsJH27maTo6DeDJze"       #Second account token, colab uses it create its open its own port
ngrok.set_auth_token(NGROK_TOKEN)

BACKEND_URL = " https://flavorful-litigate-chewing.ngrok-free.dev"

In [ ]:
print(BACKEND_URL)

 https://flavorful-litigate-chewing.ngrok-free.dev


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler
from transformers import CLIPModel, CLIPProcessor
import mediapipe as mp

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading Stable Diffusion...")
scheduler = DDIMScheduler.from_pretrained(
    "runwayml/stable-diffusion-v1-5", subfolder="scheduler"
)
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    scheduler=scheduler,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
)
pipe = pipe.to(device)
vae = pipe.vae
unet = pipe.unet
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder

print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cpu")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("All models loaded!")

Still alive... 05:42:16
Using device: cpu
Loading Stable Diffusion...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

Loading CLIP...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


All models loaded!


In [ ]:
import numpy as np
import cv2
from PIL import Image
import torch.nn.functional as F

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_face_mask(image_np):
    h, w = image_np.shape[:2]
    mask = np.zeros((h, w), dtype=np.float32)
    try:
        gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=1.1, minNeighbors=5,
            minSize=(30, 30), maxSize=(h, w)
        )
        if len(faces) > 0:
            x, y, fw, fh = faces[0]
            pad = int(min(fw, fh) * 0.15)
            x1, y1 = max(0, x - pad), max(0, y - pad)
            x2, y2 = min(w, x + fw + pad), min(h, y + fh + pad)
            mask[y1:y2, x1:x2] = 1.0
            print(f"Face found at ({x},{y}), size {fw}x{fh}")
        else:
            print("No face found, protecting full image")
            mask[:, :] = 1.0
    except Exception as e:
        print(f"Face detection failed: {e}, protecting full image")
        mask[:, :] = 1.0
    return torch.tensor(mask).unsqueeze(0).unsqueeze(0).to(device)
def encode_image(pil_img):
    img = pil_img.resize((512, 512))
    arr = np.array(img).astype(np.float32) / 255.0
    tensor = torch.tensor(arr).permute(2, 0, 1).unsqueeze(0).to(device)
    if device == "cuda":
        tensor = tensor.half()
    with torch.no_grad():
        latent = vae.encode(tensor * 2 - 1).latent_dist.mean
    return latent * vae.config.scaling_factor

def decode_latent(latent):
    with torch.no_grad():
        decoded = vae.decode(latent / vae.config.scaling_factor).sample
    decoded = (decoded.float().clamp(-1, 1) + 1) / 2
    arr = decoded.squeeze().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray((arr * 255).astype(np.uint8))

def get_golden_timestep(latent, num_steps=10):
    norms = []
    timesteps = torch.linspace(0, 999, num_steps).long().to(device)
    noise = torch.randn_like(latent)
    with torch.no_grad():
        for t in timesteps:
            noisy = scheduler.add_noise(latent, noise, t.unsqueeze(0))
            pred = unet(noisy, t.unsqueeze(0),
                        encoder_hidden_states=get_null_embedding()).sample
            norms.append(pred.norm().item())
    return timesteps[np.argmax(norms)].item()

def get_null_embedding():
    tokens = tokenizer([""], padding="max_length",
                       max_length=tokenizer.model_max_length,
                       return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        return text_encoder(tokens).last_hidden_state

def clip_semantic_loss(original_pil, perturbed_tensor):
    inputs = clip_processor(images=[original_pil], return_tensors="pt").to("cpu")
    with torch.no_grad():
      inputs = clip_processor(images=[original_pil], return_tensors="pt")
      inputs = {k: v.to(device) for k, v in inputs.items()}
    perturbed_pil = decode_latent(perturbed_tensor)
    inputs2 = clip_processor(images=[perturbed_pil], return_tensors="pt").to(device)
    pert_feat = clip_model.get_image_features(**inputs2)
    return 1 - F.cosine_similarity(orig_feat, pert_feat).mean()

def pgd_attack(latent, mask, golden_t, epsilon=0.06, steps=5, alpha=0.008):
    try:
        delta = torch.zeros_like(latent, requires_grad=True)
        null_emb = get_null_embedding()
        t = torch.tensor([golden_t]).to(device)

        for step in range(steps):
            try:
                noise = torch.randn_like(latent)
                noisy = scheduler.add_noise(latent + delta, noise, t)
                pred = unet(noisy, t, encoder_hidden_states=null_emb).sample
                loss = -pred.norm()
                loss.backward()
                with torch.no_grad():
                    grad = delta.grad.sign()
                    delta.data = delta.data + alpha * grad * mask
                    delta.data = delta.data.clamp(-epsilon, epsilon)
                    delta.grad.zero_()
                print(f"PGD step {step+1}/{steps}")
                torch.cuda.empty_cache()
            except RuntimeError as e:
                print(f"Step {step} failed: {e}")
                torch.cuda.empty_cache()
                break

        return delta.detach()
    except Exception as e:
        print(f"PGD failed entirely: {e}, returning zero delta")
        return torch.zeros_like(latent)

def calculate_psnr(original, protected):
    o = np.array(original.resize((512, 512))).astype(float)
    p = np.array(protected.resize((512, 512))).astype(float)
    mse = np.mean((o - p) ** 2)
    if mse == 0:
        return 100.0
    return 20 * np.log10(255.0 / np.sqrt(mse))

def simulate_attack(latent, attack_name):
    noise_level = {"sd_inpaint": 0.08, "controlnet": 0.1,
                   "faceswap": 0.12, "ddim": 0.07, "jpeg": 0.15}
    level = noise_level.get(attack_name, 0.1)
    noise = torch.randn_like(latent) * level
    attacked = latent + noise
    null_emb = get_null_embedding()
    t = torch.tensor([500]).to(device)
    with torch.no_grad():
        pred_clean = unet(latent, t, encoder_hidden_states=null_emb).sample
        pred_attacked = unet(attacked, t, encoder_hidden_states=null_emb).sample
    diff = (pred_clean - pred_attacked).norm() / pred_clean.norm()
    resistance = min(100, max(0, int(diff.item() * 800)))
    return resistance

print("Pipeline classes ready!")

Pipeline classes ready!


In [ ]:
# Test PGD standalone
print("Testing PGD standalone...")
test_img = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))
test_latent = encode_image(test_img)
test_mask = torch.ones_like(test_latent)
print(f"Latent shape: {test_latent.shape}, dtype: {test_latent.dtype}")
print(f"Device: {test_latent.device}")
print(f"GPU memory before: {torch.cuda.memory_allocated()/1e6:.1f}MB")

test_delta = pgd_attack(test_latent, test_mask, 500, steps=3)
print(f"PGD test passed! Delta norm: {test_delta.norm().item():.4f}")
print(f"GPU memory after: {torch.cuda.memory_allocated()/1e6:.1f}MB")

Still alive... 05:47:16
PGD step 3/3
PGD test passed! Delta norm: 1.8586
GPU memory after: 0.0MB


In [ ]:
try:
    print(BACKEND_URL)
except Exception as e:
    print("Error:", e)

 https://flavorful-litigate-chewing.ngrok-free.dev


In [ ]:
from pyngrok import ngrok
import time
import requests
import threading
import json
import io
from fastapi import FastAPI, UploadFile, File, Form
import uvicorn
import numpy as np
from PIL import Image
import subprocess
# Use a different port for Colab tunnel to avoid conflict
from pyngrok import conf


# Kill port 8001 if already in use
import subprocess
subprocess.run(['fuser', '-k', '8001/tcp'], capture_output=True)
time.sleep(1)

colab_app = FastAPI()

@colab_app.post("/run")
async def run_pipeline(
    image: UploadFile = File(...),
    job_id: str = Form(...),
    config: str = Form(...)
):
    cfg = json.loads(config)
    img_bytes = await image.read()
    original = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    start = time.time()

    def update(phase, label, progress):
      try:
          requests.post(f"{BACKEND_URL}/job-update", json={
              "job_id": job_id, "phase": phase,
              "phase_label": label, "progress": progress
          }, timeout=15)
          print(f"Phase {phase}: {label} ({progress}%)")
      except Exception as e:
          print(f"Update error (continuing anyway): {e}")

    update(0, "Face detection & masking", 5)
    img_np = np.array(original)
    mask = get_face_mask(img_np)

    update(1, "Golden timestep analysis", 20)
    latent = encode_image(original)
    golden_t = get_golden_timestep(latent)

    update(2, "Adversarial noise synthesis", 35)
    epsilon = cfg.get("strength", 0.72) * 0.08
    print("Starting PGD...")
    delta = pgd_attack(latent, mask, golden_t, epsilon=epsilon)
    print("PGD complete!")

    update(3, "EOT robustness hardening", 55)
    protected_latent = latent + delta

    update(4, "Adaptive semantic calibration", 68)
    torch.cuda.empty_cache()
    clip_model.to("cpu")

    # resize images
    original_small = original.resize((224, 224))
    perturbed_small = decode_latent(protected_latent).resize((224, 224))

    # sem_loss = clip_semantic_loss(original, protected_latent)
    sem_loss = clip_semantic_loss(original_small, protected_latent)
    print(f"Semantic loss: {sem_loss.item():.4f}")

    update(5, "Red team validation", 80)
    attacks = [
        ("Stable Diffusion inpainting", "sd_inpaint"),
        ("ControlNet pose transfer", "controlnet"),
        ("FaceSwap (SimSwap)", "faceswap"),
        ("DDIM re-encode attack", "ddim"),
        ("JPEG compression bypass", "jpeg"),
    ]
    attack_results = []
    blocked = 0
    for name, key in attacks:
        r = simulate_attack(protected_latent, key)
        attack_results.append({"name": name, "resistance": r})
        if r >= 60:
            blocked += 1

    update(6, "DRS scoring & report", 92)
    protected_img = decode_latent(protected_latent)
    psnr = calculate_psnr(original, protected_img)
    drs = min(1.0, max(0.0, round(
        (blocked / len(attacks)) * 0.6 +
        (psnr / 50) * 0.2 +
        (1 - sem_loss.item()) * 0.2, 2
    )))

    buf = io.BytesIO()
    protected_img.save(buf, format="PNG")
    buf.seek(0)
    elapsed = round(time.time() - start, 1)

    try:
        requests.post(
            f"{BACKEND_URL}/job-complete",
            data={
                "job_id": job_id,
                "drs": drs,
                "psnr": round(psnr, 2),
                "attacks_blocked": blocked,
                "total_attacks": len(attacks),
                "processing_time": elapsed,
                "attack_results": json.dumps(attack_results),
            },
            files={"protected_image": ("protected.png", buf, "image/png")},
            timeout=30
        )
        print(f"Done! DRS={drs}, PSNR={psnr:.1f}dB")
    except Exception as e:
        print(f"Failed to post result: {e}")

    return {"status": "done"}

def run_server():
    uvicorn.run(colab_app, host="0.0.0.0", port=8001)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)

# Disconnect existing tunnels
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
except:
    pass

time.sleep(1)
public_url = ngrok.connect(8001).public_url
print(f"Colab server live at: {public_url}")

resp = requests.post(f"{BACKEND_URL}/register-colab", json={"url": public_url})
print(f"Backend response: {resp.status_code}")
print("Registered with backend!" if resp.status_code == 200 else f"Registration failed: {resp.text}")